# ch06 Bonus 01：额外实验——分类 token 选择与输入长度

> 对照官方 `ch06/02_bonus_additional-experiments`

## 一句话

主线用「最后一个 token」做分类。本 notebook 实验两个问题：
1. **用第一个 token 代替最后一个，效果差多少？**
2. **输入长度（padding 多少）对分类有影响吗？**

## 背景

因果注意力下，**后面的 token 能看到前面的，前面的看不到后面的**。所以最后一个 token 信息最全，理论上最适合分类。但第一个 token（如 BERT 的 `[CLS]`）在某些双向架构里也常用——在单向 GPT 里会怎样？实验见分晓。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import tiktoken
from src.gpt import GPTModel, GPT_CONFIG_124M

# 复用主线的 demo 数据
POSITIVE = ["这部电影非常精彩 我很喜欢", "太好看了 剧情感人至深",
            "画面优美 值得推荐", "演技出色 故事动人",
            "完美之作 强烈推荐", "音乐动听 视觉震撼",
            "节奏紧凑 引人入胜", "结局温暖 回味无穷"]
NEGATIVE = ["太糟糕了 浪费时间", "剧情无聊 让人失望",
            "画面粗糙 毫无诚意", "演技尴尬 故事混乱",
            "简直烂片 不忍直视", "噪音刺耳 看不下去",
            "节奏拖沓 昏昏欲睡", "结局糟糕 一无是处"]
texts = POSITIVE + NEGATIVE
labels = [1]*len(POSITIVE) + [0]*len(NEGATIVE)
tok = tiktoken.get_encoding("gpt2")

class SentiDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len, pad_id=50256):
        self.max_len, self.pad_id = max_len, pad_id
        self.data = [(tokenizer.encode(t)[:max_len], l) for t, l in zip(texts, labels)]
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        ids, l = self.data[i]
        ids = ids + [self.pad_id]*(self.max_len-len(ids))
        return torch.tensor(ids), torch.tensor(l)

In [ ]:
def build_and_train(use_last_token, max_len, epochs=12):
    """训练一个分类器，返回训练准确率。use_last_token 决定用哪个 token。"""
    cfg = dict(GPT_CONFIG_124M)
    cfg.update({"emb_dim":128, "n_layers":2, "n_heads":4, "context_length":max_len})
    torch.manual_seed(123)
    model = GPTModel(cfg)
    model.out_head = nn.Linear(cfg["emb_dim"], 2)
    for p in model.parameters(): p.requires_grad = False
    for p in model.trf_blocks[-1].parameters(): p.requires_grad = True
    for p in model.final_norm.parameters(): p.requires_grad = True
    for p in model.out_head.parameters(): p.requires_grad = True
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-4)
    dl = DataLoader(SentiDataset(texts, labels, tok, max_len), batch_size=4, shuffle=True)
    model.train()
    for ep in range(epochs):
        for x, y in dl:
            opt.zero_grad()
            logits = model(x)
            # ★ 实验变量：用最后 token 还是第一个 token
            token_pos = -1 if use_last_token else 0
            loss = F.cross_entropy(logits[:, token_pos, :], y)
            loss.backward(); opt.step()
    # 评估
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in dl:
            pred = model(x)[:, token_pos, :].argmax(-1)
            correct += (pred==y).sum().item(); total += len(y)
    return correct/total

## 实验 1：最后 token vs 第一个 token

In [ ]:
max_len = 32
print(f"{'分类 token':<12} {'准确率':<10}")
print("-" * 24)
for use_last, name in [(True, "最后 token"), (False, "第一个 token")]:
    acc = build_and_train(use_last, max_len)
    print(f"{name:<12} {100*acc:.0f}%")
print("\n💡 因果注意力下，最后一个 token 看过全句，信息最全，分类更准。")
print("   第一个 token 看不到后续内容，在单向模型里做分类天然吃亏。")

## 实验 2：不同输入长度的影响

In [ ]:
print(f"{'输入长度':<10} {'最后token准确率':<16}")
print("-" * 28)
for ml in [16, 32, 48, 64]:
    acc = build_and_train(use_last_token=True, max_len=ml)
    print(f"{ml:<10} {100*acc:.0f}%")
print("\n💡 demo 数据句子很短，超过实际长度后全是 padding，影响不大。")
print("   真实场景（长影评）中，足够的输入长度能容纳完整信息，分类更稳。")